In [ ]:
!uv pip install pymupdf litellm sentence-transformers

# pymupdf : edit pdfs
# litellm
import pymupdf
import numpy as np
import json
import os
from litellm import completion, embedding



os.environ['GROQ_API_KEY'] = ""


Using Python 3.14.6 environment at: /Users/divyansh/Developer/AI-Ganga/.venv
Checked 3 packages in 51ms


In [32]:
# extracting text from pdf : we extract text from a pdf file using pymupdf . this process involves opening the pdf
# reading its context and converting them into a format suitable for further processing 

def extract_text_from_pdf(pdf_path):
    """ 
    extracts and consolidates text from all pages of a PDF file. This is the first step in a RAG pipeline 
    where we acquire the raw textual data that will later be processed , embedded and retrieved against .


    Args:
    pdf_path(str) : Path to the pdf file to be processed

    Returns:
    str: Complete extracted text from all pages of the PDF , concatenated into a single string .
        This raw text will be further processed in subsequent steps of the RAG pipeline 


    """

    mypdf= pymupdf.open(pdf_path)
    all_text = ""

    # iteratre through each page in the pdf

    for page_num in range(mypdf.page_count): # number of pages 
        page = mypdf[page_num]
        text = page.get_text("text")
        all_text += text

    return all_text


In [33]:
# chunking the extracted text :
# once we have the extracted text , we divide it into smaller , overlapping chunks to improve
# retrieval accuracy 


def chunk_text(text , n , overlap):
    """
    Divide text into smaller . overlapping chunks for more effective processing 
    Chunking is a critical step in RAG systems as it :
    1. its good


    The overlap between chunks helps maintain context continuity and reduces the risk of 
    splitting important information across chunk boundaries
    """
    chunks = []
    # loop through the text with a step size of (n - overlap)
    for i in range(0, len(text) , n - overlap):
        chunks.append(text[i:i + n])
    return chunks

In [34]:
path = "/Users/divyansh/Desktop/Notes/Spring.pdf"
# 1. Assign the extracted text to a variable
text = extract_text_from_pdf(path)

# 2. Pass the 'text' variable instead of the function
text_chunks = chunk_text(text, 1000, 200)

print("Number of text Chunks:", len(text_chunks))
print("\n First Chunk text:")
print(text_chunks[0])


Number of text Chunks: 1014

 First Chunk text:
M A N N I N G
LAURENŢIU SPILCĂ
FOREWORD BY VICTOR RENTEA
LEARN WHAT YOU NEED 
AND LEARN IT WELL
  
Business logic code
Transactions
Security
Logging
Caching
Data transfer
Data persistence
Use the Spring IoC container to
manage object instances easier
and glue in other functionalities
Spring provides.
Use Spring Data to connect to 
the SQL and NoSQL databases
your backend app uses to persist
the data. 
Use Spring Integration or
Spring for Apache Kafka to
more easily send messages
to your JMS or Kafka topics. 
Use Spring Security to implement
the authentication and authorization
configurations.
Use Spring Boot to ease the 
complexity of your configurations 
and write less code to implement 
the app.
Use Spring MVC or Spring WebFlux
to implement the REST endpoints
called by the client apps or other
backend solutions.
Figure 1
The user’s perspective is similar 
to viewing an iceberg. Users mainly observe 
the results of the business logic cod

In [35]:
from sentence_transformers import SentenceTransformer

# Load a local model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

def create_embeddings(text_chunks):
    """
    Transforms text into dense vector representations (embeddings) using a local neural network model.
    """
    # This runs locally and doesn't require any API key
    embeddings = embedding_model.encode(text_chunks)
    return embeddings

response = create_embeddings(text_chunks)
print("Created embeddings with shape:", response.shape)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8908.41it/s]


Created embeddings with shape: (1014, 384)


In [36]:
#performing semantic search 
# we implement cosine similarity to find the most relevant text chunks for a user 

def cosine_similarity(vec1 , vec2):
    """
    Calculates the cosine similarity between two vectors which measures the cosine of the angle 
    between them .
    Cosine similarity for RAG :
    1. It measures semantic similarity independent of vector magnitude (document length)
    2. It ranges from -1 ( completely opposite ) to 1 ( exactly the same) , making it easy to interpret 
    3. It works well in high - dimensional space like those used for text embeddings
    4. It's computationally efficient compared to some other similarity metrics


    Args:

    vec1 (np.ndarray): The first embedding vector.
    vec2 (np.ndarray): The second embedding vector.

    Returns:
    float: The cosine similarity score between the two vectors, ranging from -1 to 1.
           Higher values indicate greater semantic similarity between the original texts.
    """
    # compute the dot product of the two vectors and divide by the product of their norms 
    return np.dot(vec1 , vec2) / (np.linalg.norm(vec1)* np.linalg.norm(vec2))

In [37]:
def semantic_search(query, text_chunks, embeddings, k=5):
    """
    Performs semantic search to find the most relevant text chunks for a given query.
    """
    # Create an embedding for the query (1D numpy array)
    query_embedding = create_embeddings(query)
    similarity_scores = []

    # Calculate similarity scores between the query embedding and each chunk embedding
    for i, chunk_embedding in enumerate(embeddings):
        similarity_score = cosine_similarity(query_embedding, chunk_embedding)
        similarity_scores.append((i, similarity_score))

    # Sort the similarity scores in descending order
    similarity_scores.sort(key=lambda x: x[1], reverse=True)
    # Get the indices of the top k most similar text chunks
    top_indices = [index for index, _ in similarity_scores[:k]]
    # Return the top k most relevant text chunks
    return [text_chunks[index] for index in top_indices]


In [38]:
# Your question
query = "What is Spring Boot?"

# Perform semantic search
top_chunks = semantic_search(
    query,
    text_chunks,
    response,
    k=2
)

# Print the query
print("Query:", query)

# Print the top 2 relevant text chunks
for i, chunk in enumerate(top_chunks):
    print(f"Context {i + 1}:\n{chunk}\n=====================================")


Query: What is Spring Boot?
Context 1:
oupId>org.springframework.boot</groupId>
   <artifactId>spring-boot-starter-web</artifactId>
</dependency>
5. The properties ﬁle
application.properties
application.yml
Figure 7.13
When generating a Spring Boot project with Spring Initializr, it makes some configurations to the 
project that you don’t find in a plain Maven project.
This annotation defines the Main 
class of a Spring Boot app.
167
The magic of Spring Boot
Spring Initializr generated all this code. In this book, we’ll only focus on what’s rele-
vant to our examples. For example, I won’t detail what the SpringApplication.run()
method does and how precisely Spring Boot uses the @SpringBootApplication anno-
tation. These details aren’t relevant to what you’re learning now. Spring Boot is a sub-
ject for a whole book. But at some point you’ll undoubtedly want to understand how
Spring Boot apps work in detail, and for this I recommend you read Craig Walls’s
Spring Boot in Action (Manning,

In [40]:
# Define the system prompt for the AI assistant
system_prompt = "You are an AI assistant that strictly answers based on the given context. If the answer cannot be derived directly from the provided context, respond with: 'I do not have enough information to answer that.'"

def generate_response(system_prompt, user_message, model="groq/openai/gpt-oss-120b"):
    """
    Generates a contextually informed response using an LLM on Groq with the retrieved information.
    """
    response = completion(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ],
        temperature=0
    )
    return response

# Create the user prompt based on the top retrieved chunks
user_prompt = "\n".join([f"Context {i + 1}:\n{chunk}\n=====================================\n" for i, chunk in enumerate(top_chunks)])
user_prompt = f"{user_prompt}\nQuestion: {query}"

# Generate AI response using Groq
ai_response = generate_response(system_prompt, user_prompt)

# Print the generated answer
print("\n--- AI Response ---\n")
print(ai_response.choices[0].message.content)



--- AI Response ---

Spring Boot is a core project in the Spring ecosystem that makes it easier to create and run Spring‑based applications. It provides conventions, starter dependencies, auto‑configuration and embedded servers so developers can quickly set up a Spring app without the extensive manual configuration required in a plain Maven project. It is widely used by teams to simplify the implementation of Spring applications.
